In [ ]:

import pandas as pd

# 定义训练集文件路径
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/train.csv'

# 读取训练数据
train_df = pd.read_csv(train_data_path)

# 查看数据集的前几行
train_df.head()


Displayed on the user's machine.

In [ ]:

# 查看数据集的整体情况
print(train_df.info())

# 检查每一列的缺失值情况
print(train_df.isnull().sum())

# 检查目标变量 'Status' 的唯一值
print('Unique values in Status:', train_df['Status'].unique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6324 entries, 0 to 6323
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             6324 non-null   int64  
 1   N_Days         6324 non-null   int64  
 2   Drug           6324 non-null   object 
 3   Age            6324 non-null   int64  
 4   Sex            6324 non-null   object 
 5   Ascites        6324 non-null   object 
 6   Hepatomegaly   6324 non-null   object 
 7   Spiders        6324 non-null   object 
 8   Edema          6324 non-null   object 
 9   Bilirubin      6324 non-null   float64
 10  Cholesterol    6324 non-null   float64
 11  Albumin        6324 non-null   float64
 12  Copper         6324 non-null   float64
 13  Alk_Phos       6324 non-null   float64
 14  SGOT           6324 non-null   float64
 15  Tryglicerides  6324 non-null   float64
 16  Platelets      6324 non-null   float64
 17  Prothrombin    6324 non-null   float64
 18  Stage   

In [ ]:


# 定义测试集文件路径
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/test.csv'

# 读取测试数据
test_df = pd.read_csv(test_data_path)

# 查看测试数据集的前几行
test_df.head()



Displayed on the user's machine.

In [ ]:


from sklearn.preprocessing import LabelEncoder

# 定义需要编码的分类特征
categorical_features = ['Sex', 'Drug', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Status']

# 创建LabelEncoder实例
encoder = LabelEncoder()

# 对训练集和测试集中的分类特征进行编码
for feature in categorical_features:
    train_df[feature] = encoder.fit_transform(train_df[feature])
    test_df[feature] = encoder.fit_transform(test_df[feature])

# 查看编码后的训练数据集的前几行
train_df.head()




Displayed on the user's machine.

In [ ]:


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

# 定义特征和目标变量
X = train_df.drop(['id', 'Status'], axis=1)
y = train_df['Status']

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 初始化逻辑回归模型
model = LogisticRegression(max_iter=1000)

# 训练模型
model.fit(X_train, y_train)

# 预测验证集
y_val_pred = model.predict_proba(X_val)

# 计算验证集的logloss
val_logloss = log_loss(y_val, y_val_pred)
print(f'Validation LogLoss: {val_logloss:.4f}')

# 预测测试集
X_test = test_df.drop(['id', 'Status'], axis=1)
y_test = test_df['Status']
y_test_pred = model.predict_proba(X_test)

# 计算测试集的logloss
test_logloss = log_loss(y_test, y_test_pred)
print(f'Test LogLoss: {test_logloss:.4f}')



Validation LogLoss: 0.6008
Test LogLoss: 0.6041
D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:



# 重新初始化逻辑回归模型，增加最大迭代次数
model = LogisticRegression(max_iter=5000)

# 重新训练模型
model.fit(X_train, y_train)

# 预测验证集
y_val_pred = model.predict_proba(X_val)

# 计算验证集的logloss
val_logloss = log_loss(y_val, y_val_pred)
print(f'Validation LogLoss (after increasing max_iter): {val_logloss:.4f}')

# 预测测试集
y_test_pred = model.predict_proba(X_test)

# 计算测试集的logloss
test_logloss = log_loss(y_test, y_test_pred)
print(f'Test LogLoss (after increasing max_iter): {test_logloss:.4f}')




Validation LogLoss (after increasing max_iter): 0.5710
Test LogLoss (after increasing max_iter): 0.5646
D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:


from sklearn.preprocessing import StandardScaler

# 初始化StandardScaler
scaler = StandardScaler()

# 对训练集和验证集的特征进行标准化
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 重新初始化逻辑回归模型，增加最大迭代次数
model = LogisticRegression(max_iter=5000)

# 重新训练模型
model.fit(X_train_scaled, y_train)

# 预测验证集
y_val_pred = model.predict_proba(X_val_scaled)

# 计算验证集的logloss
val_logloss = log_loss(y_val, y_val_pred)
print(f'Validation LogLoss (after scaling features): {val_logloss:.4f}')

# 预测测试集
y_test_pred = model.predict_proba(X_test_scaled)

# 计算测试集的logloss
test_logloss = log_loss(y_test, y_test_pred)
print(f'Test LogLoss (after scaling features): {test_logloss:.4f}')




Validation LogLoss (after scaling features): 0.5359
Test LogLoss (after scaling features): 0.5296
